[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-08-measuring-generators-and-gan-seeds.ipynb)

# Advanced Discussion 8 — measuring generators, and GANs across seeds

Reproduces the chapter 8 advanced discussion: modes, quality, Fréchet distance and precision/recall on hand-made generators, then non-saturating GAN versus WGAN-GP over three seeds.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: 2-D toy data. The GAN section takes roughly 5 minutes on CPU; comment out seeds to shorten it.

In [ ]:
%pip install -q scipy

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time
from scipy import linalg

## 1. the data: 8 Gaussians on a ring (a classic mode-collapse test bed)

In [ ]:
def sample_real(n, rng):
    ang = rng.randint(0, 8, n) * (2 * np.pi / 8)
    centers = 2.0 * np.c_[np.cos(ang), np.sin(ang)]
    return (centers + 0.05 * rng.randn(n, 2)).astype(np.float32)
CENTERS = 2.0 * np.c_[np.cos(np.arange(8) * 2 * np.pi / 8), np.sin(np.arange(8) * 2 * np.pi / 8)]

## 2. metrics: modes covered, high-quality fraction, Frechet distance, k-NN precision / recall

In [ ]:
def modes_and_quality(fake, thr=0.5):
    d = np.linalg.norm(fake[:, None, :] - CENTERS[None], axis=2)
    near = d.min(1) < thr; assign = d.argmin(1)
    counts = np.bincount(assign[near], minlength=8)
    return int((counts >= 0.01 * len(fake)).sum()), float(near.mean())
def frechet(a, b):
    m1, m2 = a.mean(0), b.mean(0); c1, c2 = np.cov(a.T), np.cov(b.T)
    s = linalg.sqrtm(c1 @ c2).real
    return float(((m1 - m2) ** 2).sum() + np.trace(c1 + c2 - 2 * s))
def prec_recall(real, fake, k=3):
    def radii(x): d = np.linalg.norm(x[:, None] - x[None], axis=2); return np.sort(d, 1)[:, k]
    rr, rf = radii(real), radii(fake)
    dfr = np.linalg.norm(fake[:, None] - real[None], axis=2)      # fake x real
    precision = float((dfr <= rr[None]).any(1).mean())            # fake points inside the real manifold
    recall = float((dfr.T <= rf[None]).any(1).mean())             # real points inside the fake manifold
    return precision, recall
rng = np.random.RandomState(0)
real_a, real_b = sample_real(1000, rng), sample_real(1000, rng)
print("real vs real: modes %d, quality %.2f, Frechet %.4f, precision/recall %s" % (*modes_and_quality(real_a), frechet(real_a, real_b), tuple(round(v, 2) for v in prec_recall(real_a, real_b))))
blob = rng.randn(1000, 2).astype(np.float32) * 0.5 + 0.0
print("a single Gaussian blob: modes %d, quality %.2f, Frechet %.3f, precision/recall %s" % (*modes_and_quality(blob), frechet(real_a, blob), tuple(round(v, 2) for v in prec_recall(real_a, blob))))
one = (CENTERS[0] + 0.05 * rng.randn(1000, 2)).astype(np.float32)
print("collapsed onto ONE mode: modes %d, quality %.2f, Frechet %.3f, precision/recall %s" % (*modes_and_quality(one), frechet(real_a, one), tuple(round(v, 2) for v in prec_recall(real_a, one))))

matched = rng.multivariate_normal(real_a.mean(0), np.cov(real_a.T), 1000).astype(np.float32)       # a Gaussian with the SAME mean and covariance
print("moment-matched Gaussian: modes %d, quality %.2f, Frechet %.4f, precision/recall %s" % (*modes_and_quality(matched), frechet(real_a, matched), tuple(round(v, 2) for v in prec_recall(real_a, matched))))

## 3. GANs: non-saturating loss versus WGAN-GP

In [ ]:
def G_net(): return nn.Sequential(nn.Linear(8, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
def D_net(): return nn.Sequential(nn.Linear(2, 128), nn.LeakyReLU(0.2), nn.Linear(128, 128), nn.LeakyReLU(0.2), nn.Linear(128, 1))
def train(kind, seed, iters=6000, bs=128):
    torch.manual_seed(seed); r = np.random.RandomState(seed)
    G, D = G_net(), D_net()
    if kind == "nonsat":
        oG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999)); oD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
    else:
        oG = torch.optim.Adam(G.parameters(), 1e-4, betas=(0.5, 0.9)); oD = torch.optim.Adam(D.parameters(), 1e-4, betas=(0.5, 0.9))
    for it in range(iters):
        for _ in range(1 if kind == "nonsat" else 5):
            real = torch.tensor(sample_real(bs, r)); z = torch.randn(bs, 8); fake = G(z).detach()
            if kind == "nonsat":
                lossD = F.binary_cross_entropy_with_logits(D(real), torch.ones(bs, 1)) + F.binary_cross_entropy_with_logits(D(fake), torch.zeros(bs, 1))
            else:
                eps = torch.rand(bs, 1); xh = (eps * real + (1 - eps) * fake).requires_grad_(True)
                g = torch.autograd.grad(D(xh).sum(), xh, create_graph=True)[0]
                gp = ((g.norm(dim=1) - 1) ** 2).mean()
                lossD = D(fake).mean() - D(real).mean() + 10 * gp
            oD.zero_grad(); lossD.backward(); oD.step()
        z = torch.randn(bs, 8)
        lossG = F.binary_cross_entropy_with_logits(D(G(z)), torch.ones(bs, 1)) if kind == "nonsat" else -D(G(z)).mean()
        oG.zero_grad(); lossG.backward(); oG.step()
    with torch.no_grad(): return G(torch.randn(2000, 8)).numpy()
import sys
if '--metrics-only' in sys.argv: sys.exit()
real = sample_real(2000, np.random.RandomState(123))
rows = {}
for kind in ("nonsat", "wgan-gp"):
    res = []
    for seed in (0, 1, 2):
        t0 = time.time(); fake = train(kind, seed)
        m, q = modes_and_quality(fake); p, rc = prec_recall(real[:1000], fake[:1000])
        res.append((m, q, frechet(real, fake), p, rc)); print("  %s seed %d: modes %d/8, quality %.2f, Frechet %.3f, precision %.2f, recall %.2f (%.0fs)" % (kind, seed, m, q, res[-1][2], p, rc, time.time() - t0))
    rows[kind] = np.mean(res, axis=0)
for k, v in rows.items(): print("MEAN %-8s modes %.1f/8, quality %.2f, Frechet %.3f, precision %.2f, recall %.2f" % (k, *v))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.